# Lesson 7 — Linear Layer From Scratch

## 学习目标

前面已经学习了：

- Tensor 与 Shape Thinking；
- Broadcasting；
- Matrix Multiplication；
- Einsum；
- Autograd；
- `nn.Module` 与 `nn.Parameter`。

这一节第一次把这些知识组合起来，实现一个真正的神经网络 Layer：

`Linear`

Linear Layer 是 Transformer 中最基础、使用最频繁的组件之一。

例如：

- Query Projection；
- Key Projection；
- Value Projection；
- Attention Output Projection；
- Feed-Forward Network。

完成本节后，应能够：

1. 理解 Linear Layer 的数学形式；
2. 推导单个向量经过 Linear 后的 Shape；
3. 推导 Batch 输入经过 Linear 后的 Shape；
4. 推导 Transformer `(B,T,D)` 输入经过 Linear 后的 Shape；
5. 理解 Weight 与 Bias 的 Shape；
6. 理解 PyTorch 为什么把 Linear Weight 存储为 `(D_out,D_in)`；
7. 使用 `nn.Module` 和 `nn.Parameter` 自己实现 Linear；
8. 理解 Bias 如何通过 Broadcasting 加到输出；
9. 使用 Autograd 验证 Linear 的 Gradient；
10. 与官方 `nn.Linear` 进行 Reference Test。

这一节最终要自己实现：

`class Linear(nn.Module)`

核心计算：

$$
Y=XW^T+b
$$

并能够完整解释：

$$
Input\ Shape
$$

$$
Parameter\ Shape
$$

$$
Output\ Shape
$$

以及：

$$
Gradient\ Shape
$$


## 1. Linear Transformation

Linear Layer 最基本的数学形式可以写成：

$$
y=xW+b
$$

假设输入：

$$
x\in\mathbb{R}^{D_{in}}
$$

Weight：

$$
W\in\mathbb{R}^{D_{in}\times D_{out}}
$$

Bias：

$$
b\in\mathbb{R}^{D_{out}}
$$

那么输出：

$$
y\in\mathbb{R}^{D_{out}}
$$

从 Shape 的角度：

$$
x.shape=(D_{in})
$$

$$
W.shape=(D_{in},D_{out})
$$

矩阵乘法：

$$
(D_{in})(D_{in},D_{out})
\rightarrow
(D_{out})
$$

然后加 Bias：

$$
(D_{out})+(D_{out})
\rightarrow
(D_{out})
$$

最终：

$$
y.shape=(D_{out})
$$

因此 Linear Layer 最核心的作用可以理解为：

> 把一个 $D_{in}$ 维向量映射成一个 $D_{out}$ 维向量。

也就是：

$$
\mathbb{R}^{D_{in}}
\rightarrow
\mathbb{R}^{D_{out}}
$$


In [1]:
import torch

D_in = 4
D_out = 6

x = torch.randn(D_in)
W = torch.randn(D_in, D_out)
b = torch.randn(D_out)

y = x @ W + b

print("x.shape:", x.shape)
print("W.shape:", W.shape)
print("b.shape:", b.shape)
print("y.shape:", y.shape)

x.shape: torch.Size([4])
W.shape: torch.Size([4, 6])
b.shape: torch.Size([6])
y.shape: torch.Size([6])


## 2. 从单个输出元素理解 Linear

假设：

$$
x\in\mathbb{R}^{D_{in}}
$$

经过 Linear 后：

$$
y\in\mathbb{R}^{D_{out}}
$$

输出中的第 $j$ 个元素：

$$
y_j
$$

可以写成：

$$
y_j
=
\sum_{i=1}^{D_{in}}
x_iW_{ij}
+
b_j
$$

也就是说：

> 每一个输出 feature，都会使用输入向量的所有 feature。

例如：

$$
D_{in}=4
$$

那么一个输出元素可能是：

$$
y_1
=
x_1W_{11}
+
x_2W_{21}
+
x_3W_{31}
+
x_4W_{41}
+
b_1
$$

因此 Linear Layer 并不是简单地逐元素缩放。

它会把输入 feature 进行线性组合。

如果：

$$
D_{out}=6
$$

那么模型会产生 6 个不同的线性组合：

$$
y_1,y_2,\dots,y_6
$$

最终组成：

$$
y\in\mathbb{R}^{6}
$$


In [2]:
D_in = 4
D_out = 6

x = torch.randn(D_in)
W = torch.randn(D_in, D_out)
b = torch.randn(D_out)

y = x @ W + b

y0_manual = x[0] * W[0, 0] + x[1] * W[1, 0] + x[2] * W[2, 0] + x[3] * W[3, 0] + b[0]

print("Linear output y[0]:", y[0])
print("Manual output:", y0_manual)
print("same:", torch.allclose(y[0], y0_manual))

Linear output y[0]: tensor(4.7266)
Manual output: tensor(4.7266)
same: True


## 3. Batch Linear

神经网络通常不会一次只处理一个样本。

假设：

$$
X.shape=(B,D_{in})
$$

其中：

- $B$：Batch Size；
- $D_{in}$：输入 Feature Dimension。

Weight：

$$
W.shape=(D_{in},D_{out})
$$

计算：

$$
Y=XW
$$

根据矩阵乘法：

$$
(B,D_{in})
(D_{in},D_{out})
\rightarrow
(B,D_{out})
$$

然后加入 Bias：

$$
b.shape=(D_{out})
$$

利用 Broadcasting：

$$
(B,D_{out})
+
(D_{out})
\rightarrow
(B,D_{out})
$$

最终：

$$
Y.shape=(B,D_{out})
$$

重要的是：

> Batch 中所有样本共享同一个 Weight 和 Bias。

也就是说：

$$
y_1=x_1W+b
$$

$$
y_2=x_2W+b
$$

$$
\vdots
$$

$$
y_B=x_BW+b
$$

PyTorch 使用一次矩阵运算：

`X @ W + b`

就可以同时完成整个 Batch 的 Linear Transformation。


In [3]:
B = 3
D_in = 4
D_out = 6

X = torch.randn(B, D_in)
W = torch.randn(D_in, D_out)
b = torch.randn(D_out)

Y = X @ W + b

print("X.shape:", X.shape)
print("W.shape:", W.shape)
print("b.shape:", b.shape)
print("Y.shape:", Y.shape)

X.shape: torch.Size([3, 4])
W.shape: torch.Size([4, 6])
b.shape: torch.Size([6])
Y.shape: torch.Size([3, 6])


## 4. Batch Linear 的真实含义

对于：

$$
X.shape=(B,D_{in})
$$

执行：

$$
Y=XW+b
$$

可以理解为：

> 对 Batch 中每一行分别执行同一个 Linear Transformation。

例如：

$$
X=
\begin{bmatrix}
x_1\\
x_2\\
x_3
\end{bmatrix}
$$

那么：

$$
Y=
\begin{bmatrix}
x_1W+b\\
x_2W+b\\
x_3W+b
\end{bmatrix}
$$

所以 Linear Layer 不会在不同 Batch 样本之间进行信息混合。

Batch 维只是：

> 多个样本并行计算。

这一点之后理解 Transformer 也非常重要。


In [4]:
B = 3
D_in = 4
D_out = 6

X = torch.randn(B, D_in)
W = torch.randn(D_in, D_out)
b = torch.randn(D_out)

Y_batch = X @ W + b
Y_manual = torch.stack([X[i] @ W + b for i in range(B)])

print("Y_batch.shape:", Y_batch.shape)
print("Y_manual.shape:", Y_manual.shape)
print("same:", torch.allclose(Y_batch, Y_manual))

Y_batch.shape: torch.Size([3, 6])
Y_manual.shape: torch.Size([3, 6])
same: True


## 5. Transformer 中的 Linear

Transformer 的输入通常不是：

$$
(B,D)
$$

而是：

$$
X.shape=(B,T,D)
$$

其中：

- $B$：Batch Size；
- $T$：Sequence Length；
- $D$：Model / Hidden Dimension。

如果 Linear 的输入维度为：

$$
D_{in}
$$

输出维度为：

$$
D_{out}
$$

那么：

$$
X.shape=(B,T,D_{in})
$$

Weight：

$$
W.shape=(D_{in},D_{out})
$$

计算：

$$
Y=XW
$$

Shape：

$$
(B,T,D_{in})
(D_{in},D_{out})
\rightarrow
(B,T,D_{out})
$$

再加 Bias：

$$
(D_{out})
$$

通过 Broadcasting：

$$
(B,T,D_{out})
+
(D_{out})
\rightarrow
(B,T,D_{out})
$$

所以最终：

$$
Y.shape=(B,T,D_{out})
$$

可以理解为：

> 对所有 Batch 中的所有 Token，独立应用同一个 Linear Transformation。

Linear 不会改变：

- Batch 数量 $B$；
- Token 数量 $T$。

它改变的是最后一个 Feature Dimension：

$$
D_{in}
\rightarrow
D_{out}
$$


In [5]:
B = 2
T = 8
D_in = 16
D_out = 32

X = torch.randn(B, T, D_in)
W = torch.randn(D_in, D_out)
b = torch.randn(D_out)

Y = X @ W + b

print("X.shape:", X.shape)
print("W.shape:", W.shape)
print("b.shape:", b.shape)
print("Y.shape:", Y.shape)

X.shape: torch.Size([2, 8, 16])
W.shape: torch.Size([16, 32])
b.shape: torch.Size([32])
Y.shape: torch.Size([2, 8, 32])


## 6. Linear 不会混合 Token

这是理解 Transformer 时非常重要的一点。

假设：

$$
X.shape=(B,T,D)
$$

执行 Linear：

$$
Y=XW+b
$$

对于某一个 Token：

$$
X[b,t,:]
$$

输出：

$$
Y[b,t,:]
$$

只依赖：

$$
X[b,t,:]
$$

本身。

它不会直接依赖：

$$
X[b,s,:]
$$

其中：

$$
s\neq t
$$

所以 Linear 做的是：

> Feature Dimension 上的变换。

而不是：

> Token Dimension 上的信息交互。

Shape：

$$
(B,T,D_{in})
\rightarrow
(B,T,D_{out})
$$

其中：

$$
T
$$

保持不变。

Transformer 中真正让不同 Token 相互交互的主要机制是 Self-Attention。

因此可以建立一个重要区别：

Linear：

$$
Feature\ Mixing
$$

Self-Attention：

$$
Token\ Mixing
$$

这个区别后面理解 Transformer Block 非常重要。


In [6]:
B = 2
T = 3
D_in = 4
D_out = 6

X = torch.randn(B, T, D_in)
W = torch.randn(D_in, D_out)
b = torch.randn(D_out)

Y_full = X @ W + b
Y_token = X[0, 1] @ W + b

print("full result:", Y_full[0, 1])
print("single token:", Y_token)
print("same:", torch.allclose(Y_full[0, 1], Y_token))

full result: tensor([ 0.1559, -1.7700,  0.6800, -0.4305,  0.3231, -1.7555])
single token: tensor([ 0.1559, -1.7700,  0.6800, -0.4305,  0.3231, -1.7555])
same: True


## 7. Linear 的通用 Shape Rule

前面的例子：

单个向量：

$$
(D_{in})
\rightarrow
(D_{out})
$$

Batch：

$$
(B,D_{in})
\rightarrow
(B,D_{out})
$$

Transformer：

$$
(B,T,D_{in})
\rightarrow
(B,T,D_{out})
$$

实际上可以统一成：

$$
(...,D_{in})
\rightarrow
(...,D_{out})
$$

这里：

$$
...
$$

表示任意数量的前置维度。

例如：

$$
(2,3,4,16)
$$

经过：

$$
Linear(16,32)
$$

得到：

$$
(2,3,4,32)
$$

所以判断 Linear 输出 Shape 时，可以形成条件反射：

> 前面的所有维度保持不变，只把最后一个 `in_features` 替换成 `out_features`。

即：

$$
(...,D_{in})
\rightarrow
(...,D_{out})
$$


In [7]:
D_in = 16
D_out = 32

X = torch.randn(2, 3, 4, D_in)
W = torch.randn(D_in, D_out)
b = torch.randn(D_out)

Y = X @ W + b

print("X.shape:", X.shape)
print("Y.shape:", Y.shape)

X.shape: torch.Size([2, 3, 4, 16])
Y.shape: torch.Size([2, 3, 4, 32])


## 8. PyTorch `nn.Linear`

PyTorch 已经提供了官方 Linear Layer：

`nn.Linear`

基本形式：

`nn.Linear(in_features, out_features)`

例如：

`nn.Linear(4, 6)`

表示：

$$
D_{in}=4
$$

$$
D_{out}=6
$$

因此输入：

$$
(...,4)
$$

输出：

$$
(...,6)
$$

例如：

$$
(2,8,4)
$$

经过：

`nn.Linear(4, 6)`

得到：

$$
(2,8,6)
$$

但是这里有一个非常重要的实现细节：

> PyTorch 内部保存的 Weight Shape 与我们前面的数学记法方向相反。


In [8]:
from torch import nn

linear = nn.Linear(4, 6)

x = torch.randn(2, 8, 4)
y = linear(x)

print("input:", x.shape)
print("output:", y.shape)

input: torch.Size([2, 8, 4])
output: torch.Size([2, 8, 6])


## 9. PyTorch 为什么保存 `(out_features, in_features)`？

前面为了方便理解矩阵乘法，我们使用：

$$
Y=XW+b
$$

其中：

$$
W.shape=(D_{in},D_{out})
$$

但是 PyTorch 的：

`nn.Linear(in_features, out_features)`

内部保存：

$$
weight.shape
=
(D_{out},D_{in})
$$

例如：

`nn.Linear(4, 6)`

内部：

$$
weight.shape=(6,4)
$$

Bias：

$$
bias.shape=(6)
$$

因此 PyTorch 的数学形式通常写成：

$$
Y=XW^T+b
$$

因为：

$$
W.shape=(D_{out},D_{in})
$$

转置后：

$$
W^T.shape=(D_{in},D_{out})
$$

于是：

$$
(...,D_{in})
(D_{in},D_{out})
\rightarrow
(...,D_{out})
$$

所以：

$$
Y=XW^T+b
$$

与前面使用的：

$$
Y=XW+b
$$

本质上是同一个 Linear Transformation。

区别只是：

> Weight Parameter 在内


## 10. PyTorch Linear 的 Forward Shape

假设输入：

$$
X.shape=(B,T,D_{in})
$$

PyTorch 保存的 Weight：

$$
W.shape=(D_{out},D_{in})
$$

如果直接计算：

`X @ W`

最后两个相关维度会变成：

$$
D_{in}
$$

和：

$$
D_{out}
$$

它们通常不相等，因此不能进行我们想要的矩阵乘法。

所以需要先转置 Weight：

$$
W^T.shape=(D_{in},D_{out})
$$

然后：

$$
XW^T
$$

Shape 推导：

$$
(B,T,D_{in})
(D_{in},D_{out})
\rightarrow
(B,T,D_{out})
$$

最后加入：

$$
b.shape=(D_{out})
$$

利用 Broadcasting：

$$
(B,T,D_{out})
+
(D_{out})
\rightarrow
(B,T,D_{out})
$$

因此 PyTorch Linear 的核心计算可以理解为：

$$
Y=XW^T+b
$$

其中：

$$
W.shape=(D_{out},D_{in})
$$

这是我们接下来自己实现 Linear 时必须保持一致的 Weight 存储方式。


In [9]:
B = 2
T = 3
D_in = 4
D_out = 6

X = torch.randn(B, T, D_in)
W = torch.randn(D_out, D_in)
b = torch.randn(D_out)

print("X.shape:", X.shape)
print("W.shape:", W.shape)
print("W.T.shape:", W.T.shape)

Y = X @ W.T + b

print("Y.shape:", Y.shape)

X.shape: torch.Size([2, 3, 4])
W.shape: torch.Size([6, 4])
W.T.shape: torch.Size([4, 6])
Y.shape: torch.Size([2, 3, 6])


## 11. Linear From Scratch

现在已经具备实现 Linear 所需要的全部知识。

我们的目标：

`class Linear(nn.Module)`

需要保存两个模型参数。

### Weight

$$
weight.shape=(D_{out},D_{in})
$$

### Bias

$$
bias.shape=(D_{out})
$$

因为它们都是需要训练的参数，所以必须使用：

`nn.Parameter`

而不是普通 Tensor。

因此模型结构为：

Linear

↓

Weight Parameter

$$
(D_{out},D_{in})
$$

↓

Bias Parameter

$$
(D_{out})
$$

Forward 则执行：

$$
Y=XW^T+b
$$

输入允许具有任意数量的前置维度：

$$
(...,D_{in})
$$

输出：

$$
(...,D_{out})
$$


In [10]:
class Linear(nn.Module):
    def __init__(self, in_features: int, out_features: int) -> None:
        super().__init__()

        self.in_features = in_features
        self.out_features = out_features

        self.weight = nn.Parameter(torch.empty(out_features, in_features))
        self.bias = nn.Parameter(torch.empty(out_features))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x @ self.weight.T + self.bias

## 12. 检查 Parameter Shape

创建：

`Linear(4, 6)`

表示：

$$
D_{in}=4
$$

$$
D_{out}=6
$$

因此应该得到：

$$
weight.shape=(6,4)
$$

以及：

$$
bias.shape=(6)
$$

这与：

`nn.Linear(4, 6)`

的 Parameter Shape 保持一致。

同时，因为：

`weight`

和：

`bias`

都是：

`nn.Parameter`

所以它们应该自动出现在：

`layer.named_parameters()`

中。

这是 Lesson 6 学习的 Parameter Registration。


In [11]:
layer = Linear(4, 6)

print("in_features:", layer.in_features)
print("out_features:", layer.out_features)
print("weight.shape:", layer.weight.shape)
print("bias.shape:", layer.bias.shape)

in_features: 4
out_features: 6
weight.shape: torch.Size([6, 4])
bias.shape: torch.Size([6])


## 13. Parameter Registration

我们的代码：

`self.weight = nn.Parameter(...)`

以及：

`self.bias = nn.Parameter(...)`

会让 PyTorch 自动注册这两个 Parameter。

因此：

`layer.named_parameters()`

应该能够找到：

- `weight`
- `bias`

这非常重要。

因为未来：

`model.parameters()`

会把这些 Parameter 交给 Optimizer。

如果 Parameter 没有正确注册，那么即使它参与 Forward 和 Autograd，也可能不会被标准模型训练流程正确管理。

所以实现自定义 Layer 时，一个基本检查就是：

> `named_parameters()` 是否包含所有应该训练的 Parameter？


In [12]:
layer = Linear(4, 6)

for name, parameter in layer.named_parameters():
    print(name, parameter.shape)


weight torch.Size([6, 4])
bias torch.Size([6])


## 14. Forward Shape Test

现在测试：

$$
X.shape=(B,T,D_{in})
$$

例如：

$$
X.shape=(2,8,16)
$$

创建：

`Linear(16, 32)`

Parameter：

$$
weight.shape=(32,16)
$$

$$
bias.shape=(32)
$$

Forward：

$$
XW^T+b
$$

首先：

$$
W^T.shape=(16,32)
$$

所以：

$$
(2,8,16)
(16,32)
\rightarrow
(2,8,32)
$$

然后 Bias Broadcasting：

$$
(2,8,32)
+
(32)
\rightarrow
(2,8,32)
$$

因此最终：

$$
output.shape=(2,8,32)
$$


In [13]:
B = 2
T = 8
D_in = 16
D_out = 32

layer = Linear(D_in, D_out)

x = torch.randn(B, T, D_in)
y = layer(x)

print("input:", x.shape)
print("weight:", layer.weight.shape)
print("bias:", layer.bias.shape)
print("output:", y.shape)

input: torch.Size([2, 8, 16])
weight: torch.Size([32, 16])
bias: torch.Size([32])
output: torch.Size([2, 8, 32])


## 15. `torch.empty()` 并没有初始化参数

目前我们创建 Parameter 时使用：

`torch.empty(...)`

例如：

`torch.empty(32, 16)`

需要注意：

> `torch.empty()` 只负责分配内存，不会把 Tensor 初始化成有意义的数值。

也就是说：

`empty`

不是：

`zeros`

也不是：

`randn`

其中原本存在什么内存数据，就可能看到什么数值。

因此我们的 Linear 虽然：

- Parameter Shape 正确；
- Parameter Registration 正确；
- Forward 公式正确；

但是还缺少：

> Parameter Initialization

一个真正可以正常使用的 Linear Layer，必须在创建 Weight 和 Bias 后进行初始化。

所以完整结构应该变成：

`__init__`

↓

创建 Parameter

↓

`reset_parameters()`

↓

初始化 Weight / Bias

↓

Forward


In [14]:
layer = Linear(4, 6)

print("weight:")
print(layer.weight)
print()
print("bias:")
print(layer.bias)

weight:
Parameter containing:
tensor([[-1.8807e+00,  7.0065e-45,  0.0000e+00,  0.0000e+00],
        [ 7.9472e-32,  3.3002e-41,  5.9655e+29,  4.0640e-41],
        [ 7.9472e-32,  3.3002e-41,  1.2826e+19,  2.3051e-12],
        [ 2.8699e-42,  0.0000e+00,  3.5873e-43,  1.0385e+34],
        [ 6.2920e+29,  4.0640e-41,  0.0000e+00,  0.0000e+00],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00]],
       requires_grad=True)

bias:
Parameter containing:
tensor([-1.7779e-07,  3.2995e-41,  0.0000e+00,  0.0000e+00,  4.4842e-44,
         0.0000e+00], requires_grad=True)


## 16. Parameter Initialization

一个自然的问题是：

> 既然 `torch.empty()` 不行，为什么不直接全部初始化成 0？

例如：

$$
W=0
$$

对于单个简单 Linear Layer，数学运算本身当然可以执行。

但是在深层神经网络中，如果多个神经元以完全相同的参数开始：

- Forward 行为相同；
- 接收到的 Gradient 可能具有相同结构；
- 它们难以学习出不同的特征。

这通常称为：

> Symmetry Problem

因此神经网络 Weight 通常需要随机初始化。

但随机值也不能随便设置得过大或过小。

如果 Weight Scale 不合适：

- Activation 可能越来越大；
- Activation 可能越来越小；
- Gradient 也可能变得不稳定。

因此出现了专门的初始化策略，例如：

- Xavier Initialization；
- Kaiming Initialization。

这一节不深入推导初始化理论。

当前目标只是建立：

> Trainable Parameter 创建之后，需要合理初始化。


## 17. Xavier Initialization

这里先使用：

`nn.init.xavier_uniform_`

初始化 Weight。

对于：

$$
W.shape=(D_{out},D_{in})
$$

Xavier Initialization 会根据：

$$
fan_{in}
$$

和：

$$
fan_{out}
$$

控制初始化数值范围。

当前不要求记住具体公式。

只需要知道：

> 初始化范围会根据输入维度和输出维度进行调整。

Bias 暂时初始化为：

$$
0
$$

因此：

Weight

→ Xavier Uniform

Bias

→ Zero

后面如果需要严格复现某种模型，我们会根据该模型的初始化规则重新调整。


In [15]:
layer = Linear(16, 32)

with torch.no_grad():
    nn.init.xavier_uniform_(layer.weight)

    nn.init.zeros_(layer.bias)

print("weight mean:", layer.weight.mean().item())
print("bias:", layer.bias[:5])

weight mean: 0.015381600707769394
bias: tensor([0., 0., 0., 0., 0.], grad_fn=<SliceBackward0>)


## 18. Initialization 与 Autograd

Parameter Initialization 发生在训练开始之前。

例如：

`nn.init.xavier_uniform_(weight)`

只是设置 Parameter 的初始数值。

我们不需要计算：

> 初始化操作本身对 Loss 的 Gradient。

因此初始化属于：

Model Setup

而不是：

Training Computation

所以通常不会让这些初始化操作成为 Computational Graph 的一部分。

PyTorch 的 `torch.nn.init` 初始化函数本身就是以不参与 Autograd 的方式修改 Tensor。

这里使用：

`torch.no_grad()`

可以帮助我们继续强化一个概念：

> 参数的手动数值修改，不应该被当成模型 Forward 的一部分。


## 19. `reset_parameters()`

初始化不应该要求使用者每次创建：

`Linear(...)`

之后再手动执行。

更合理的设计是：

在 Module 内部定义：

`reset_parameters()`

然后在：

`__init__()`

中自动调用。

结构：

`__init__`

↓

创建 Weight

↓

创建 Bias

↓

`reset_parameters()`

这样：

`Linear(16, 32)`

创建完成后就已经拥有有效的初始 Parameter。

这也是 PyTorch Module 中很常见的设计模式。


In [16]:
class Linear(nn.Module):
    def __init__(self, in_features: int, out_features: int) -> None:

        super().__init__()

        self.in_features = in_features
        self.out_features = out_features

        self.weight = nn.Parameter(torch.empty(out_features, in_features))
        self.bias = nn.Parameter(torch.empty(out_features))

        self.reset_parameters()

    def reset_parameters(self) -> None:

        nn.init.xavier_uniform_(self.weight)
        nn.init.zeros_(self.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:

        return x @ self.weight.T + self.bias


In [17]:
layer = Linear(4, 6)

x = torch.randn(2, 3, 4)
y = layer(x)

print("input:", x.shape)
print("weight:", layer.weight.shape)
print("bias:", layer.bias.shape)
print("output:", y.shape)
print()
print("weight mean:", layer.weight.mean().item())
print("bias:", layer.bias)

input: torch.Size([2, 3, 4])
weight: torch.Size([6, 4])
bias: torch.Size([6])
output: torch.Size([2, 3, 6])

weight mean: 0.06151514872908592
bias: Parameter containing:
tensor([0., 0., 0., 0., 0., 0.], requires_grad=True)


## 21. Shape Correct 不代表 Implementation Correct

目前我们的实现已经满足：

### Parameter Shape

$$
weight.shape=(D_{out},D_{in})
$$

$$
bias.shape=(D_{out})
$$

### Forward Shape

$$
(...,D_{in})
\rightarrow
(...,D_{out})
$$

### Parameter Registration

`weight`

和：

`bias`

都出现在：

`named_parameters()`

中。

但是：

> Shape 正确，并不能证明数值计算一定正确。

例如一个错误实现也可能碰巧输出相同 Shape。

所以实现基础神经网络组件时，需要建立一个非常重要的测试习惯：

> 与可信的 Reference Implementation 对齐。

我们的 Reference：

`torch.nn.Linear`

下一步需要：

1. 创建我们的 Linear；
2. 创建官方 `nn.Linear`；
3. 把两边的 Weight 和 Bias 设置成完全相同；
4. 输入完全相同的 Tensor；
5. 比较 Forward Output；
6. 再比较 Backward Gradient。

只有这样才能更有信心地说明实现正确。


In [18]:
torch.manual_seed(0)

my_linear = Linear(4, 6)

reference_linear = nn.Linear(4, 6)

print("my weight:", my_linear.weight.shape)
print("reference weight:", reference_linear.weight.shape)
print("my bias:", my_linear.bias.shape)
print("reference bias:", reference_linear.bias.shape)

my weight: torch.Size([6, 4])
reference weight: torch.Size([6, 4])
my bias: torch.Size([6])
reference bias: torch.Size([6])


## 22. Forward Reference Test

现在比较：

我们的：

`Linear`

和官方：

`nn.Linear`

为了公平比较，必须保证两边：

- Weight 完全相同；
- Bias 完全相同；
- 输入完全相同。

否则即使实现完全正确，因为初始化不同，输出也不会相同。

因此测试步骤：

1. 创建两个 Linear；
2. 把官方 Linear 的 Weight 复制成我们的 Weight；
3. 把 Bias 也复制过去；
4. 输入同一个 Tensor；
5. 比较输出。

如果：

`torch.allclose(...)`

返回：

`True`

说明 Forward 数值实现与官方 Linear 一致。


In [19]:
torch.manual_seed(0)

my_linear = Linear(4, 6)
reference_linear = nn.Linear(4, 6)

with torch.no_grad():
    reference_linear.weight.copy_(my_linear.weight)
    reference_linear.bias.copy_(my_linear.bias)

x = torch.randn(2, 3, 4)

my_output = my_linear(x)
reference_output = reference_linear(x)

print("my output shape:", my_output.shape)
print("reference output shape:", reference_output.shape)
print("same output:", torch.allclose(my_output, reference_output))

my output shape: torch.Size([2, 3, 6])
reference output shape: torch.Size([2, 3, 6])
same output: True


## 23. Reference Test 的意义

如果只检查：

$$
output.shape=(B,T,D_{out})
$$

仍然可能存在很多错误。

例如：

- Weight 转置方向写错；
- Bias 加错位置；
- 使用了错误 Parameter；
- 某个维度被错误交换；
- 数值运算公式写错。

这些错误有时仍然可能产生“看起来合理”的 Shape。

因此应该区分：

### Shape Test

检查：

> Tensor 维度是否正确。

### Numerical Reference Test

检查：

> 实际数值计算是否和可信实现一致。

实现 Transformer Primitive 时，后面会反复使用：

自己实现

↓

Reference Implementation

↓

相同 Input / Parameter

↓

比较 Output

这种测试方式。


## 24. Forward 正确还不够

Linear Layer 最终需要训练。

所以不仅 Forward 必须正确。

Backward 也必须能够得到正确 Gradient。

需要检查三个 Gradient：

### Input Gradient

$$
\frac{\partial L}{\partial X}
$$

### Weight Gradient

$$
\frac{\partial L}{\partial W}
$$

### Bias Gradient

$$
\frac{\partial L}{\partial b}
$$

因为我们的实现只使用标准 PyTorch Tensor Operations：

- Matrix Multiplication；
- Transpose；
- Addition；

所以 Autograd 理论上能够自动生成正确 Backward。

但是工程上仍然应该实际验证。


In [20]:
torch.manual_seed(0)

my_linear = Linear(4, 6)


reference_linear = nn.Linear(4, 6)

with torch.no_grad():
    reference_linear.weight.copy_(my_linear.weight)
    reference_linear.bias.copy_(my_linear.bias)

x1 = torch.randn(2, 3, 4, requires_grad=True)
x2 = x1.clone().detach()
x2.requires_grad_(True)

my_output = my_linear(x1)
reference_output = reference_linear(x2)

my_loss = my_output.pow(2).mean()
reference_loss = reference_output.pow(2).mean()

my_loss.backward()
reference_loss.backward()

print("input gradient same:", torch.allclose(x1.grad, x2.grad))
print(
    "weight gradient same:",
    torch.allclose(my_linear.weight.grad, reference_linear.weight.grad),
)
print(
    "bias gradient same:",
    torch.allclose(my_linear.bias.grad, reference_linear.bias.grad),
)


input gradient same: True
weight gradient same: True
bias gradient same: True


## 25. Linear 的 Gradient Shape

假设：

$$
X.shape=(B,T,D_{in})
$$

$$
W.shape=(D_{out},D_{in})
$$

$$
b.shape=(D_{out})
$$

Forward：

$$
Y=XW^T+b
$$

最终得到 scalar loss：

$$
L
$$

Backward 后：

### Input Gradient

$$
X.grad.shape
=
(B,T,D_{in})
$$

### Weight Gradient

$$
W.grad.shape
=
(D_{out},D_{in})
$$

### Bias Gradient

$$
b.grad.shape
=
(D_{out})
$$

再次形成：

> 一个 Tensor 的 Gradient，需要告诉我们该 Tensor 中每一个元素应该如何变化。

所以 Gradient Shape 通常和对应 Tensor Shape 相同。


In [21]:
layer = Linear(16, 32)

x = torch.randn(2, 8, 16, requires_grad=True)
y = layer(x)

loss = y.pow(2).mean()
loss.backward()

print("x.shape:", x.shape)
print("x.grad.shape:", x.grad.shape)
print("weight.shape:", layer.weight.shape)
print("weight.grad.shape:", layer.weight.grad.shape)
print("bias.shape:", layer.bias.shape)
print("bias.grad.shape:", layer.bias.grad.shape)

x.shape: torch.Size([2, 8, 16])
x.grad.shape: torch.Size([2, 8, 16])
weight.shape: torch.Size([32, 16])
weight.grad.shape: torch.Size([32, 16])
bias.shape: torch.Size([32])
bias.grad.shape: torch.Size([32])


## 26. Parameter Count

Linear Layer 包含：

### Weight

$$
W.shape=
(D_{out},D_{in})
$$

Parameter 数量：

$$
D_{out}D_{in}
$$

### Bias

$$
b.shape=(D_{out})
$$

Parameter 数量：

$$
D_{out}
$$

因此总参数量：

$$
N
=
D_{out}D_{in}
+
D_{out}
$$

也可以写成：

$$
N
=
D_{out}(D_{in}+1)
$$

如果：

`bias=False`

则：

$$
N=D_{out}D_{in}
$$

注意：

Batch Size：

$$
B
$$

以及 Sequence Length：

$$
T
$$

都不会影响 Linear 的参数量。

因为所有 Batch 和 Token 共享同一组 Weight。


In [22]:
D_in = 768
D_out = 3072

weight_parameters = D_out * D_in
bias_parameters = D_out
total_parameters = weight_parameters + bias_parameters

print("weight parameters:", weight_parameters)
print("bias parameters:", bias_parameters)
print("total parameters:", total_parameters)

weight parameters: 2359296
bias parameters: 3072
total parameters: 2362368


In [23]:
layer = Linear(768, 3072)

manual_count = 768 * 3072 + 3072
module_count = sum(parameter.numel() for parameter in layer.parameters())

print("manual:", manual_count)
print("module:", module_count)
print("same:", manual_count == module_count)

manual: 2362368
module: 2362368
same: True


## 28. Transformer 中的 Linear Projection

Transformer 中很多看起来不同的操作，本质上都是：

> 对每一个 token 的最后一个 feature dimension 做 Linear Projection。

输入：

$$
X.shape=(B,T,D)
$$

---

### Query Projection

$$
Q=XW_Q^T
$$

通常：

$$
Q.shape=(B,T,D)
$$

---

### Key Projection

$$
K=XW_K^T
$$

通常：

$$
K.shape=(B,T,D)
$$

---

### Value Projection

$$
V=XW_V^T
$$

通常：

$$
V.shape=(B,T,D)
$$

---

### Attention Output Projection

Multi-Head Attention 合并 Heads 后：

$$
(B,T,D)
$$

再经过：

$$
Linear(D,D)
$$

输出仍然：

$$
(B,T,D)
$$

---

所以 Attention 中至少会出现多个 Linear Projection。

Linear 是 Transformer 最重要的基础计算之一。


## 29. FFN 中的 Linear

Transformer Block 除了 Attention，还有 Feed-Forward Network。

一种简化结构：

$$
X
\rightarrow
Linear(D,D_{ff})
\rightarrow
Activation
\rightarrow
Linear(D_{ff},D)
$$

其中通常：

$$
D_{ff}>D
$$

例如：

$$
D_{ff}=4D
$$

那么第一层：

$$
(B,T,D)
\rightarrow
(B,T,4D)
$$

第二层：

$$
(B,T,4D)
\rightarrow
(B,T,D)
$$

注意整个过程中：

$$
B
$$

和：

$$
T
$$

始终没有改变。

Linear 改变的仍然只是最后一个 Feature Dimension。


## 30. 一个非常重要的概念

对于：

$$
X.shape=(B,T,D)
$$

普通 Linear：

$$
Linear(D_{in},D_{out})
$$

独立作用于：

$$
X[b,t,:]
$$

也就是说：

> Linear 会混合 Feature Dimension，但不会直接混合不同 Token Position。

对于 token：

$$
t_1
$$

它的 Linear Output 只取决于：

$$
X[b,t_1,:]
$$

不会直接读取：

$$
X[b,t_2,:]
$$

而 Self-Attention 的作用则不同。

Attention 会建立：

$$
(T,T)
$$

的 token-to-token relation。

因此可以形成一个非常重要的区分：

### Linear

主要混合：

$$
Feature
$$

### Attention

主要建立：

$$
Token
\leftrightarrow
Token
$$

交互。

这两种操作组合起来，构成 Transformer 的核心计算模式。


In [24]:
torch.manual_seed(0)

layer = Linear(4, 6)

x = torch.randn(1, 3, 4)

y1 = layer(x)

x_modified = x.clone()
x_modified[0, 0] += 10

y2 = layer(x_modified)

print("token 0 changed:", not torch.allclose(y1[0, 0], y2[0, 0]))
print("token 1 unchanged:", torch.allclose(y1[0, 1], y2[0, 1]))
print("token 2 unchanged:", torch.allclose(y1[0, 2], y2[0, 2]))

token 0 changed: True
token 1 unchanged: True
token 2 unchanged: True


## 32. Optional Bias

并不是所有 Linear 都必须使用 Bias。

数学上可以使用：

$$
Y=XW^T
$$

而没有：

$$
+b
$$

因此我们可以让构造函数支持：

`bias: bool = True`

如果：

`bias=True`

创建：

$$
b\in\mathbb{R}^{D_{out}}
$$

如果：

`bias=False`

则不创建 Bias Parameter。

PyTorch Module 中可以使用：

`register_parameter("bias", None)`

明确表示：

> 这个位置没有 Parameter。

这样接口就更加接近官方：

`nn.Linear`


In [25]:
class Linear(nn.Module):
    def __init__(self, in_features: int, out_features: int, bias: bool = True) -> None:
        super().__init__()

        self.in_features = in_features
        self.out_features = out_features

        self.weight = nn.Parameter(torch.empty(out_features, in_features))

        if bias:
            self.bias = nn.Parameter(torch.empty(out_features))

        else:
            self.register_parameter("bias", None)

        self.reset_parameters()

    def reset_parameters(self) -> None:

        nn.init.xavier_uniform_(self.weight)

        if self.bias is not None:
            nn.init.zeros_(self.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:

        output = x @ self.weight.T

        if self.bias is not None:
            output = output + self.bias

        return output


In [26]:
layer = Linear(4, 6, bias=False)

x = torch.randn(2, 3, 4)
y = layer(x)

print("output:", y.shape)
print("bias:", layer.bias)
print()

for name, parameter in layer.named_parameters():
    print(name, parameter.shape)


output: torch.Size([2, 3, 6])
bias: None

weight torch.Size([6, 4])


## 35. 常见错误

### 错误 1：把 Weight Shape 写反

如果按照 PyTorch 风格存储：

$$
weight.shape=(D_{out},D_{in})
$$

那么 Forward 应该使用：

$$
XW^T
$$

不能忘记转置。

---

### 错误 2：数学中的 W 和 PyTorch 保存的 weight 混淆

数学常写：

$$
W.shape=(D_{in},D_{out})
$$

PyTorch `nn.Linear` 保存：

$$
weight.shape=(D_{out},D_{in})
$$

二者只是存储约定不同。

---

### 错误 3：使用普通 Tensor 保存 Weight

错误：

`self.weight = torch.empty(...)`

这样它不会自动成为注册 Parameter。

应该：

`self.weight = nn.Parameter(...)`

---

### 错误 4：使用 `torch.empty()` 后忘记初始化

`torch.empty()` 只分配内存。

必须执行 Parameter Initialization。

---

### 错误 5：以为 Batch Size 会增加参数量

不会。

所有 Batch Sample 共用同一个 Weight。

---

### 错误 6：以为 Sequence Length 会增加参数量

不会。

所有 Token Position 也共用同一个 Linear。

---

### 错误 7：手动给每个 Token 写循环

不需要。

对于：

$$
(B,T,D_{in})
$$

PyTorch Matmul 可以直接得到：

$$
(B,T,D_{out})
$$

---

### 错误 8：Bias Shape 写成 `(B,T,D_out)`

不需要。

Bias 只需要：

$$
(D_{out})
$$

Broadcasting 会自动作用到 Batch 和 Token 维度。

---

### 错误 9：只测试 Shape，不测试数值

Shape 正确不代表实现正确。

应该和：

`nn.Linear`

做 Reference Test。

---

### 错误 10：只测试 Forward，不测试 Backward

自定义神经网络 Primitive 最终需要训练。

所以最好同时检查：

- output；
- input gradient；
- weight gradient；
- bias gradient。

---

### 错误 11：认为 Linear 会让 Token 相互通信

不会。

普通 Linear 独立作用于每个 token 的 feature vector。

Token-to-token communication 主要由 Attention 完成。

---

### 错误 12：认为 `loss.backward()` 会更新 Weight

不会。

它只计算：

`weight.grad`

真正的 Parameter Update 由后面的 Optimizer 完成。


## Lesson 7 总结

### Rule 1：Linear 的数学形式

常见数学写法：

$$
Y=XW+b
$$

---

### Rule 2：PyTorch Weight Storage

PyTorch 风格：

$$
weight.shape=(D_{out},D_{in})
$$

所以 Forward：

$$
Y=XW^T+b
$$

---

### Rule 3：Linear 改变最后一个维度

输入：

$$
(...,D_{in})
$$

输出：

$$
(...,D_{out})
$$

---

### Rule 4：Transformer Shape

$$
(B,T,D_{in})
$$

经过 Linear：

$$
(B,T,D_{out})
$$

其中：

$$
B
$$

和：

$$
T
$$

保持不变。

---

### Rule 5：Weight Parameter

$$
weight.shape=
(D_{out},D_{in})
$$

---

### Rule 6：Bias Parameter

$$
bias.shape=
(D_{out})
$$

利用 Broadcasting：

$$
(...,D_{out})
+
(D_{out})
$$

---

### Rule 7：Parameter Registration

使用：

`nn.Parameter`

后，Weight 和 Bias 会出现在：

`model.parameters()`

中。

---

### Rule 8：Parameter Initialization

`torch.empty()`

不是有效初始化。

Parameter 创建之后需要初始化。

---

### Rule 9：Reference Test

正确实现应该同时检查：

- Shape；
- Forward Numerical Result；
- Backward Gradient。

---

### Rule 10：Gradient Shape

$$
weight.grad.shape
=
weight.shape
$$

$$
bias.grad.shape
=
bias.shape
$$

---

### Rule 11：Parameter Count

有 Bias：

$$
N
=
D_{out}D_{in}
+
D_{out}
$$

无 Bias：

$$
N
=
D_{out}D_{in}
$$

---

### Rule 12：Linear 不混合 Token

对于：

$$
(B,T,D)
$$

Linear 独立作用于每一个：

$$
X[b,t,:]
$$

它主要进行：

$$
Feature\ Mixing
$$

而不是：

$$
Token\ Mixing
$$


## Linear Debug Checklist

以后自己实现 Linear 或其它 Projection 时，如果出现问题，可以按下面顺序检查。

### 1. Input 最后一个维度是否正确？

应该满足：

$$
x.shape[-1]
=
in\_features
$$

---

### 2. Weight Shape 是否正确？

PyTorch 风格：

$$
(D_{out},D_{in})
$$

---

### 3. Forward 是否使用了 Weight Transpose？

应该得到：

$$
(D_{in},D_{out})
$$

---

### 4. Bias Shape 是否正确？

$$
(D_{out})
$$

---

### 5. Parameter 是否注册？

检查：

`named_parameters()`

---

### 6. Parameter 是否已经初始化？

不要把未初始化的：

`torch.empty()`

直接用于正式 Forward。

---

### 7. Output Shape 是否正确？

$$
(...,D_{in})
\rightarrow
(...,D_{out})
$$

---

### 8. 是否和 Reference Forward 对齐？

使用：

`torch.allclose(...)`

---

### 9. Gradient 是否存在？

检查：

`weight.grad`

和：

`bias.grad`

---

### 10. Gradient Shape 是否正确？

应该分别与 Parameter Shape 相同。

---

### 11. Backward 是否和 Reference 对齐？

比较：

- Input Gradient；
- Weight Gradient；
- Bias Gradient。

---

### 12. 是否错误地把 Batch / Token 维度当成参数维度？

Linear Parameters 只由：

$$
D_{in}
$$

和：

$$
D_{out}
$$

决定。
